# Bank Loan & Credit Risk Analysis with SQL

Analysis of ~38,000 bank loans from the Kaggle *Financial Loan* dataset: who borrows, who defaults, and what that means for lending strategy.

**Skills shown:** conditional aggregation (`CASE WHEN`), subqueries, CTEs, window functions (`LAG`, `RANK`) and SQL views for KPI reporting.
**Stack:** Python · pandas · SQLite · Jupyter

## Contents
0. Load the data and build the database
1. Segment borrowers by income, grade and purpose (`CASE WHEN`)
2. Default rate, funded amount and interest rate trends: high-risk categories
3. Good loans vs bad loans and month-by-month performance (subqueries and CTEs)
4. KPI summary views for credit risk and lending strategy

**Definitions used throughout**
- **Default** = `loan_status = 'Charged Off'`
- **Good loan** = *Fully Paid* or *Current*; **Bad loan** = *Charged Off*
- `int_rate` and `dti` are stored as fractions (`0.1227` = 12.27 %), so queries multiply by 100 for display

### About the SQL dialect
The original script was written for **MySQL** (kept in `sql/loan_analysis_mysql.sql`). So that anyone can run this notebook without installing a database server, it runs the same queries on **SQLite**, which ships with Python. Only a few functions needed translating:

| MySQL | SQLite (this notebook) |
|---|---|
| `STR_TO_DATE(issue_date, '%d-%m-%Y')` | `substr()` re-ordering `dd-mm-yyyy` into `yyyy-mm-dd` |
| `DATE_FORMAT(issue_dt, '%Y-%m')` | `strftime('%Y-%m', issue_dt)` |
| `CREATE OR REPLACE VIEW` | `DROP VIEW IF EXISTS` + `CREATE VIEW` |
| `CREATE DATABASE` / `USE` | one database file, `loan_analysis.db` |

`CASE WHEN`, CTEs, subqueries and window functions are identical.

## 0. Load the data and build the database

In [ ]:
# Run once if a package is missing:
# %pip install -q pandas kagglehub openpyxl

import os
import glob
import sqlite3
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

### 0.1 Get the dataset from Kaggle
The notebook first looks for a file in `./data/`. If there isn't one, it downloads the dataset with [`kagglehub`](https://github.com/Kaggle/kagglehub).
If the automatic download fails, download the file manually from the dataset page and drop it into `./data/`:
<https://www.kaggle.com/datasets/datawitharyan/financial-loan-dataset>

In [ ]:
KAGGLE_DATASET = "datawitharyan/financial-loan-dataset"   # ~38,500 loans
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)


def find_data_file(folder):
    """Return the first .csv / .xlsx / .xls file under `folder`, or None."""
    for ext in ("csv", "xlsx", "xls"):
        hits = sorted(glob.glob(os.path.join(folder, "**", f"*.{ext}"), recursive=True))
        if hits:
            return hits[0]
    return None


data_file = find_data_file(DATA_DIR)              # 1) a file already in ./data
if data_file is None:                              # 2) otherwise download from Kaggle
    import kagglehub
    data_file = find_data_file(kagglehub.dataset_download(KAGGLE_DATASET))

if data_file is None:
    raise FileNotFoundError(
        "Could not find the dataset. Download it from Kaggle and put the file in ./data/"
    )
print("Using:", data_file)

if data_file.lower().endswith(".csv"):
    raw = pd.read_csv(data_file)
else:
    raw = pd.read_excel(data_file)

raw.columns = raw.columns.str.strip().str.lower()
raw = raw.rename(columns={"loan_id": "id"})        # some Kaggle copies call the key loan_id
print("Rows, columns:", raw.shape)
raw.head()

### 0.2 Prepare the data for SQL
Checks the expected columns are present, standardises the date columns to `dd-mm-yyyy` text (the format the original MySQL script expected) and makes sure `int_rate` / `dti` are fractions.

In [ ]:
EXPECTED = [
    "id", "address_state", "application_type", "emp_length", "emp_title", "grade",
    "home_ownership", "issue_date", "last_credit_pull_date", "last_payment_date",
    "loan_status", "next_payment_date", "member_id", "purpose", "sub_grade", "term",
    "verification_status", "annual_income", "dti", "installment", "int_rate",
    "loan_amount", "total_acc", "total_payment",
]
missing = [c for c in EXPECTED if c not in raw.columns]
if missing:
    raise ValueError(
        f"Columns missing from the dataset: {missing}\n"
        f"Columns found: {list(raw.columns)}\n"
        "Rename them with raw.rename(columns={...}) in the previous cell, or edit the queries."
    )
df = raw[EXPECTED].copy()


def to_ddmmyyyy(s):
    """Return a date column as dd-mm-yyyy text, whatever form it arrived in."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.strftime("%d-%m-%Y")
    parsed = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
    if parsed.isna().sum() > s.isna().sum():       # some rows use another layout
        parsed = parsed.fillna(pd.to_datetime(s, dayfirst=True, errors="coerce"))
    return parsed.dt.strftime("%d-%m-%Y")


for col in ["issue_date", "last_credit_pull_date", "last_payment_date", "next_payment_date"]:
    df[col] = to_ddmmyyyy(df[col])

# The SQL expects fractions (0.1227 = 12.27 %)
for col in ["int_rate", "dti"]:
    if df[col].max() > 1:
        print(f"{col} looks like a percentage -> dividing by 100")
        df[col] = df[col] / 100

before = len(df)
df = df.drop_duplicates(subset="id")
print(f"Dropped {before - len(df)} duplicate ids; {len(df):,} rows ready to load")

### 0.3 Create the database and table
Same table definition as the original script, minus MySQL-only syntax. The data is loaded with pandas (the SQLite equivalent of the Workbench import wizard).

In [ ]:
DB_PATH = "loan_analysis.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)                              # start fresh on every run
conn = sqlite3.connect(DB_PATH)

conn.executescript("""
CREATE TABLE bank_loan (
    id                    INTEGER PRIMARY KEY,
    address_state         VARCHAR(5),
    application_type      VARCHAR(20),
    emp_length            VARCHAR(20),
    emp_title             VARCHAR(100),
    grade                 CHAR(1),
    home_ownership        VARCHAR(20),
    issue_date            VARCHAR(12),     -- text like 11-02-2021, converted below
    last_credit_pull_date VARCHAR(12),
    last_payment_date     VARCHAR(12),
    loan_status           VARCHAR(20),     -- Fully Paid / Current / Charged Off
    next_payment_date     VARCHAR(12),
    member_id             INT,
    purpose               VARCHAR(50),
    sub_grade             VARCHAR(5),
    term                  VARCHAR(15),
    verification_status   VARCHAR(30),
    annual_income         DECIMAL(12,2),
    dti                   DECIMAL(6,4),    -- debt-to-income ratio
    installment           DECIMAL(10,2),
    int_rate              DECIMAL(6,4),    -- 0.1227 means 12.27%
    loan_amount           INT,             -- the funded amount
    total_acc             INT,
    total_payment         INT              -- amount received back from the borrower
);
""")

df.to_sql("bank_loan", conn, if_exists="append", index=False)
conn.commit()


def run_query(sql):
    """Run a SELECT and return the result as a DataFrame (shown as a table)."""
    return pd.read_sql_query(sql, conn)


run_query("SELECT * FROM bank_loan LIMIT 5")

### 0.4 Clean the data and sanity-check the import
Convert the text date into a real date column (`issue_dt`, stored as ISO `yyyy-mm-dd`), then check the row count, the `loan_status` values and the date range.

In [ ]:
conn.executescript("""
ALTER TABLE bank_loan ADD COLUMN issue_dt DATE;

UPDATE bank_loan
SET issue_dt = substr(issue_date, 7, 4) || '-' || substr(issue_date, 4, 2) || '-' || substr(issue_date, 1, 2)
WHERE issue_date IS NOT NULL;
""")
conn.commit()

In [ ]:
run_query("""
SELECT COUNT(*) AS total_rows FROM bank_loan;
""")

In [ ]:
run_query("""
SELECT loan_status, COUNT(*) AS loans
FROM bank_loan
GROUP BY loan_status;
""")

In [ ]:
run_query("""
SELECT MIN(issue_dt) AS first_loan,
       MAX(issue_dt) AS last_loan,
       SUM(issue_dt IS NULL) AS unparsed_dates   -- should be 0
FROM bank_loan;
""")

---
## 1. Conditional aggregation (`CASE WHEN`) to segment borrowers by income, grade and loan purpose

### 1A. Borrowers by income band

In [ ]:
run_query("""
SELECT
    CASE
        WHEN annual_income <  40000 THEN '1. Under 40K'
        WHEN annual_income <  80000 THEN '2. 40K - 80K'
        WHEN annual_income < 120000 THEN '3. 80K - 120K'
        ELSE                             '4. Above 120K'
    END                                                   AS income_band,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS total_funded,
    ROUND(AVG(int_rate) * 100, 2)                         AS avg_int_rate_pct,
    SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END)                    AS defaulted_loans,
    ROUND(SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) * 100.0
          / COUNT(id), 2)                                 AS default_rate_pct
FROM bank_loan
GROUP BY income_band
ORDER BY income_band;
""")

### 1B. Loan outcomes by grade (A = safest, G = riskiest)

In [ ]:
run_query("""
SELECT
    grade,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS total_funded,
    SUM(CASE WHEN loan_status = 'Fully Paid'  THEN 1 ELSE 0 END) AS fully_paid,
    SUM(CASE WHEN loan_status = 'Current'     THEN 1 ELSE 0 END) AS current_loans,
    SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) AS charged_off
FROM bank_loan
GROUP BY grade
ORDER BY grade;
""")

### 1C. Loan purpose

In [ ]:
run_query("""
SELECT
    purpose,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS total_funded,
    ROUND(AVG(dti) * 100, 2)                              AS avg_dti_pct,
    ROUND(SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) * 100.0
          / COUNT(id), 2)                                 AS default_rate_pct
FROM bank_loan
GROUP BY purpose
ORDER BY total_funded DESC;
""")

---
## 2. Default rate, funded amount and interest rate trends to identify high-risk loan categories

### 2A. Default rate and interest rate by grade and term

In [ ]:
run_query("""
SELECT
    grade,
    term,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS funded_amount,
    ROUND(AVG(int_rate) * 100, 2)                         AS avg_int_rate_pct,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY grade, term
ORDER BY default_rate_pct DESC;
""")

### 2B. High-risk purposes
Purposes whose default rate is **above the overall portfolio default rate** (uses a subquery).

In [ ]:
run_query("""
SELECT
    purpose,
    COUNT(id)                                             AS applications,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY purpose
HAVING SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id) >
       (SELECT SUM(loan_status = 'Charged Off') * 100.0 / COUNT(*) FROM bank_loan)
ORDER BY default_rate_pct DESC;
""")

### 2C. Default rate by home ownership and verification status (groups with more than 100 loans)

In [ ]:
run_query("""
SELECT
    home_ownership,
    verification_status,
    COUNT(id)                                             AS applications,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY home_ownership, verification_status
HAVING COUNT(id) > 100
ORDER BY default_rate_pct DESC;
""")

### 2D. Top 5 states by funded amount, with their default rate

In [ ]:
run_query("""
SELECT
    address_state,
    SUM(loan_amount)                                      AS funded_amount,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY address_state
ORDER BY funded_amount DESC
LIMIT 5;
""")

---
## 3. Subqueries and CTEs to compare good loans vs. bad loans and measure portfolio performance month by month

### 3A. Good loans vs bad loans (CTE)
Good = *Fully Paid* or *Current*; Bad = *Charged Off*.

In [ ]:
run_query("""
WITH loan_class AS (
    SELECT
        *,
        CASE
            WHEN loan_status IN ('Fully Paid', 'Current') THEN 'Good Loan'
            ELSE 'Bad Loan'
        END AS loan_type
    FROM bank_loan
)
SELECT
    loan_type,
    COUNT(id)                                             AS applications,
    ROUND(COUNT(id) * 100.0 / (SELECT COUNT(*) FROM bank_loan), 2) AS pct_of_portfolio,
    SUM(loan_amount)                                      AS funded_amount,
    SUM(total_payment)                                    AS amount_received,
    ROUND(AVG(int_rate) * 100, 2)                         AS avg_int_rate_pct,
    ROUND(AVG(dti) * 100, 2)                              AS avg_dti_pct
FROM loan_class
GROUP BY loan_type;
""")

### 3B. Month-by-month portfolio performance with month-over-month growth (CTE + `LAG`)

In [ ]:
run_query("""
WITH monthly AS (
    SELECT
        strftime('%Y-%m', issue_dt)                       AS ym,
        COUNT(id)                                         AS applications,
        SUM(loan_amount)                                  AS funded_amount,
        SUM(total_payment)                                AS amount_received,
        SUM(loan_status = 'Charged Off')                  AS bad_loans
    FROM bank_loan
    GROUP BY strftime('%Y-%m', issue_dt)
)
SELECT
    ym,
    applications,
    funded_amount,
    amount_received,
    ROUND(bad_loans * 100.0 / applications, 2)            AS default_rate_pct,
    ROUND((funded_amount - LAG(funded_amount) OVER (ORDER BY ym)) * 100.0
          / LAG(funded_amount) OVER (ORDER BY ym), 2)     AS mom_funded_growth_pct
FROM monthly
ORDER BY ym;
""")

### 3C. Good vs bad funded amount by month (conditional aggregation inside a CTE)

In [ ]:
run_query("""
WITH monthly_split AS (
    SELECT
        strftime('%Y-%m', issue_dt)                       AS ym,
        SUM(CASE WHEN loan_status IN ('Fully Paid','Current') THEN loan_amount ELSE 0 END) AS good_funded,
        SUM(CASE WHEN loan_status = 'Charged Off'             THEN loan_amount ELSE 0 END) AS bad_funded
    FROM bank_loan
    GROUP BY strftime('%Y-%m', issue_dt)
)
SELECT
    ym,
    good_funded,
    bad_funded,
    ROUND(bad_funded * 100.0 / (good_funded + bad_funded), 2) AS bad_share_pct
FROM monthly_split
ORDER BY ym;
""")

### 3D. Top 3 purposes by funded amount within each grade (window function `RANK`)

In [ ]:
run_query("""
SELECT *
FROM (
    SELECT
        grade,
        purpose,
        SUM(loan_amount) AS funded_amount,
        RANK() OVER (PARTITION BY grade ORDER BY SUM(loan_amount) DESC) AS rank_in_grade
    FROM bank_loan
    GROUP BY grade, purpose
) ranked
WHERE rank_in_grade <= 3
ORDER BY grade, rank_in_grade;
""")

---
## 4. Summary views of KPIs for credit risk assessment and lending strategy
Three reusable views that could feed a dashboard (Power BI, Tableau, etc.).

In [ ]:
conn.executescript("""
DROP VIEW IF EXISTS vw_kpi_summary;
DROP VIEW IF EXISTS vw_risk_by_grade;
DROP VIEW IF EXISTS vw_monthly_trend;

-- 4A. Overall KPI view
CREATE VIEW vw_kpi_summary AS
SELECT
    COUNT(id)                                             AS total_applications,
    SUM(loan_amount)                                      AS total_funded_amount,
    SUM(total_payment)                                    AS total_amount_received,
    ROUND(AVG(int_rate) * 100, 2)                         AS avg_int_rate_pct,
    ROUND(AVG(dti) * 100, 2)                              AS avg_dti_pct,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2)               AS default_rate_pct,
    ROUND(SUM(CASE WHEN loan_status IN ('Fully Paid','Current') THEN 1 ELSE 0 END)
          * 100.0 / COUNT(id), 2)                         AS good_loan_pct
FROM bank_loan;

-- 4B. Risk view by grade
CREATE VIEW vw_risk_by_grade AS
SELECT
    grade,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS funded_amount,
    ROUND(AVG(int_rate) * 100, 2)                         AS avg_int_rate_pct,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY grade;

-- 4C. Monthly trend view (for a dashboard)
CREATE VIEW vw_monthly_trend AS
SELECT
    strftime('%Y-%m', issue_dt)                           AS ym,
    COUNT(id)                                             AS applications,
    SUM(loan_amount)                                      AS funded_amount,
    SUM(total_payment)                                    AS amount_received,
    ROUND(SUM(loan_status = 'Charged Off') * 100.0 / COUNT(id), 2) AS default_rate_pct
FROM bank_loan
GROUP BY strftime('%Y-%m', issue_dt);
""")
conn.commit()

### 4A. Overall KPIs

In [ ]:
run_query("""
SELECT * FROM vw_kpi_summary;
""")

### 4B. Risk by grade (highest default rate first)

In [ ]:
run_query("""
SELECT * FROM vw_risk_by_grade ORDER BY default_rate_pct DESC;
""")

### 4C. Monthly trend

In [ ]:
run_query("""
SELECT * FROM vw_monthly_trend ORDER BY ym;
""")

In [ ]:
conn.close()